Step 1.1 Import Libraries

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
from sklearn.metrics import precision_recall_curve, confusion_matrix, log_loss
import joblib
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings for cleaner output

Step 1.2 Import Data

In [2]:
import duckdb, pyarrow
print("DuckDB:", duckdb.__version__)
print("PyArrow:", pyarrow.__version__)


DuckDB: 1.5.1
PyArrow: 23.0.1


In [3]:
!pip install duckdb pyarrow --upgrade


In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gc

# ---------- USER SETTINGS ----------
raw_root = Path(r"D:\Medical_AI\Data\raw_data\2023_to_2025_millisec")
patient_info_path = Path(r"D:\Medical_AI\Data\patient_indexes_max_hour.csv")
keep_test_types = ['flow', 'pressure']
dt_format = '%Y-%m-%d %H:%M:%S.%f'
sample_mode = False
# -----------------------------------

# Load & clean patient info
patient_info = pd.read_csv(patient_info_path, low_memory=False)
patient_info.columns = patient_info.columns.str.strip()
patient_info['patient_number'] = patient_info['patient_number'].astype(str).str.strip()
patient_info['year'] = patient_info['year'].astype(str).str.strip()
patient_info['label'] = patient_info['label'].map({'good': 0, 'bad': 1}).astype('Int64')
patient_info['start_date'] = pd.to_datetime(patient_info['start_date'], errors='coerce')
patient_info['end_date'] = pd.to_datetime(patient_info['end_date'], errors='coerce')
patient_info['machine_year'] = patient_info['patient_number'] + "_" + patient_info['year'].astype(str)

patients = patient_info[['patient_number','year','patient_id','label','start_date','end_date']].drop_duplicates()

# storage
patient_dfs_list = []
patient_meta_list = []

# iterate patients
iter_rows = list(patients.iterrows())
if sample_mode:
    iter_rows = iter_rows[:1]

for idx, row in tqdm(iter_rows, total=len(iter_rows), desc="Patients"):
    machine_number = str(row['patient_number'])
    year = str(row['year'])
    patient_id = int(row['patient_id']) if pd.notna(row['patient_id']) else None
    label = int(row['label']) if pd.notna(row['label']) else None
    start_date = row['start_date']
    end_date = row['end_date']
    folder_prefix = f"{machine_number}_{year}"

    candidate_folders = [p for p in raw_root.rglob("*") if p.is_dir() and p.name.startswith(folder_prefix)]
    if not candidate_folders:
        print(f"Warning: no folder found for {folder_prefix} -> skipping patient_id={patient_id}")
        continue

    # ✅ Match folder to label: label 0 = good, label 1 = bad
    label_suffix = 'good' if label == 0 else 'bad'
    expected_folder_name = f"{folder_prefix}_{label_suffix}"

    chosen_folder = next(
        (f for f in candidate_folders if f.name == expected_folder_name),
        None
    )
    if chosen_folder is None:
        print(f"⚠️  Expected folder '{expected_folder_name}' not found. "
              f"Candidates: {[f.name for f in candidate_folders]} -> skipping patient_id={patient_id}")
        continue

    all_csvs = sorted(chosen_folder.glob("*.csv"))
    if not all_csvs:
        print(f"Warning: no CSV files under folder {chosen_folder} -> skipping patient_id={patient_id}")
        continue

    patient_chunks = []
    machine_year_full = chosen_folder.name

    for csv_file in all_csvs:
        try:
            df_raw = pd.read_csv(csv_file, dtype=str, low_memory=False)
        except Exception as e:
            print(f"  Error reading {csv_file.name}: {e}")
            continue

        df_raw.columns = df_raw.columns.str.strip()
        dt_col = 'Datetime' if 'Datetime' in df_raw.columns else ('datetime' if 'datetime' in df_raw.columns else None)
        if dt_col is None:
            continue

        ts = pd.to_datetime(df_raw[dt_col], format=dt_format, errors='coerce')
        if ts.isna().mean() > 0.2:
            ts = pd.to_datetime(df_raw[dt_col], errors='coerce')
        df_raw['timestamp'] = ts
        df_raw = df_raw.dropna(subset=['timestamp'])
        if df_raw.empty:
            continue

        df_raw = df_raw.rename(columns={c: c.strip().lower() for c in df_raw.columns})
        present = [c for c in df_raw.columns if c in keep_test_types]
        if not present:
            continue

        df_small = df_raw[['timestamp'] + present].copy()
        df_long = df_small.melt(id_vars=['timestamp'], value_vars=present,
                                var_name='test_type', value_name='test_value')
        df_long['test_value'] = pd.to_numeric(df_long['test_value'], errors='coerce')

        df_long['patient_id'] = patient_id
        df_long['label'] = label
        df_long['start_date'] = start_date
        df_long['end_date'] = end_date
        df_long['machine_year_full'] = machine_year_full

        patient_chunks.append(df_long)

        if sum(len(c) for c in patient_chunks) > 5_000_000:
            patient_chunks = [pd.concat(patient_chunks, ignore_index=True)]
            gc.collect()

    if not patient_chunks:
        print(f"  No usable rows for {folder_prefix} (patient_id={patient_id})")
        continue

    df_patient = pd.concat(patient_chunks, ignore_index=True)
    df_patient = df_patient.sort_values('timestamp').reset_index(drop=True)
    final_cols = ['timestamp', 'test_type', 'test_value', 'patient_id', 'label', 'start_date', 'end_date', 'machine_year_full']
    df_final = df_patient[[c for c in final_cols if c in df_patient.columns]].copy()

    patient_dfs_list.append(df_final)
    patient_meta_list.append({
        'patient_id': patient_id,
        'machine_year_full': machine_year_full,
        'n_rows': len(df_final),
        'label': label
    })

    print(f"Built patient_id={patient_id} rows={len(df_final)} label={label} machine_year_full={machine_year_full}")

summary = pd.DataFrame(patient_meta_list).sort_values('patient_id').reset_index(drop=True)
print("\nSummary:")
print(summary)

Patients:   1%|          | 1/87 [00:09<13:54,  9.70s/it]

Built patient_id=1 rows=7148924 machine_year_full=B3A20604054_2023_good


Patients:   2%|▏         | 2/87 [00:16<11:29,  8.11s/it]

Built patient_id=2 rows=5420800 machine_year_full=B3A20605023_2023_good


Patients:   3%|▎         | 3/87 [00:53<29:42, 21.22s/it]

...

Built patient_id=85 rows=8302410 machine_year_full=YSD21B03010_2023_bad


Patients:  99%|█████████▉| 86/87 [16:22<00:07,  7.12s/it]

Built patient_id=86 rows=1369896 machine_year_full=YSG22303077_2023_bad


Patients: 100%|██████████| 87/87 [16:25<00:00, 11.33s/it]

Built patient_id=87 rows=3076176 machine_year_full=YSG22303077_2024_bad

Summary:
    patient_id      machine_year_full    n_rows  label
0            1  B3A20604054_2023_good   7148924      0
1            2  B3A20605023_2023_good   5420800      0
2            3  B3A20605070_2023_good  19305604      0
3            4  B3A23701088_2025_good   7042872      0
4            5  B3A23C26164_2025_good   9887072      0
..         ...                    ...       ...    ...
82          83   YSC21603101_2023_bad   3764132      1
83          84   YSD21503317_2024_bad   5751414      1
84          85   YSD21B03010_2023_bad   8302410      1
85          86   YSG22303077_2023_bad   1369896      1
86          87   YSG22303077_2024_bad   3076176      1

[87 rows x 4 columns]


In [10]:
import pandas as pd

summary = pd.DataFrame(patient_meta_list)

pd.set_option('display.max_rows', None)
print(summary)


    patient_id      machine_year_full    n_rows  label
0            1  B3A20604054_2023_good   7148924      0
1            2  B3A20605023_2023_good   5420800      0
2            3  B3A20605070_2023_good  19305604      0
3            4  B3A23701088_2025_good   7042872      0
4            5  B3A23C26164_2025_good   9887072      0
5            6  B3A24606034_2025_good   6387902      0
6            7  B3C23619117_2025_good   9989444      0
7            8  B3C23C24308_2025_good   9793832      0
8            9  B3C23C27141_2025_good   8372776      0
9           10  B3C23C27371_2025_good   2858772      0
10          11  B3C24325207_2025_good   8599090      0
11          12  B3C24329028_2024_good   6457898      0
12          13  B3C24406068_2025_good   5951580      0
13          14  B3C24513126_2025_good   2955468      0
14          15  B3C24513203_2025_good   4982618      0
15          16  B3C24513221_2025_good   4273248      0
16          17  B3C24515251_2025_good   4239008      0
17        

In [11]:
len(patient_dfs_list)

86

In [12]:
patient_dfs_list[1].head(20)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full
0,2023-09-25 13:00:34.200,flow,52.60,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
1,2023-09-25 13:00:34.200,pressure,10.87,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
2,2023-09-25 13:00:34.400,pressure,11.47,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
3,2023-09-25 13:00:34.400,flow,127.00,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
4,2023-09-25 13:00:34.600,flow,126.70,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
5,2023-09-25 13:00:34.600,pressure,11.39,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
6,2023-09-25 13:00:34.800,flow,105.20,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
7,2023-09-25 13:00:34.800,pressure,12.30,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
8,2023-09-25 13:00:35.000,pressure,5.37,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
9,2023-09-25 13:00:35.000,flow,54.60,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good


# Remove patient #86

In [13]:
df86 = next((df for df in patient_dfs_list if int(df['patient_id'].iloc[0]) == 86), None)

if df86 is not None:
    print(f"Patient 86 — total rows: {len(df86)}")
    print(f"  ts_min  : {df86['timestamp'].min()}")
    print(f"  ts_max  : {df86['timestamp'].max()}")
    print(f"  span    : {(df86['timestamp'].max() - df86['timestamp'].min()).total_seconds() / 86400:.3f} days")
    print(f"  end_date: {df86['end_date'].iloc[0]}")
else:
    print("Patient 86 not found in patient_dfs_list")

Patient 86 not found in patient_dfs_list


In [14]:
patient_dfs_list = [df for df in patient_dfs_list if int(df['patient_id'].iloc[0]) != 86]

# verify
print(f"Total patients after drop: {len(patient_dfs_list)}")
print(f"Patient 86 still exists: {any(int(df['patient_id'].iloc[0]) == 86 for df in patient_dfs_list)}")

Total patients after drop: 86
Patient 86 still exists: False


# drop nan rows

In [16]:
import pandas as pd
import gc
from tqdm import tqdm
from pathlib import Path

OUT_SUMMARY = Path(r"D:\Medical_AI\Data\processed\clean_nan_summary.csv")

results = []

for i, df in tqdm(enumerate(patient_dfs_list, start=1), desc="Cleaning patient_dfs_list"):
    # Basic safety check
    if not isinstance(df, pd.DataFrame):
        results.append({"index": i, "rows_before": 0, "nan_rows": 0, "nan_cells": 0, "rows_after": 0, "note": "not a DataFrame"})
        continue

    rows_before = len(df)
    nan_rows = int(df.isna().any(axis=1).sum())
    nan_cells = int(df.isna().sum().sum())

    if nan_rows > 0:
        # Drop any row with at least one NaN
        df_clean = df.dropna(how='any').reset_index(drop=True)
        rows_after = len(df_clean)
        # Replace the original DataFrame in the list
        patient_dfs_list[i-1] = df_clean
        del df_clean
    else:
        rows_after = rows_before

    # Record results
    results.append({
        "index": i,
        "rows_before": rows_before,
        "nan_rows": nan_rows,
        "nan_cells": nan_cells,
        "rows_after": rows_after
    })

    # Free memory
    del df
    gc.collect()

# Summary DataFrame
summary_df = pd.DataFrame(results)

# Print full summary without truncation
pd.set_option('display.max_rows', None)
print("\nCleaned summary (full):")
print(summary_df.to_string(index=False))

# Save to CSV
try:
    OUT_SUMMARY.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(OUT_SUMMARY, index=False)
    print(f"\nSaved summary to: {OUT_SUMMARY}")
except Exception as e:
    print("Could not save summary:", e)


Cleaning patient_dfs_list: 86it [03:43,  2.60s/it]


Cleaned summary (full):
 index  rows_before  nan_rows  nan_cells  rows_after
     1      7148924     12474      12474     7136450
     2      5420800   1137770    1137770     4283030
     3     19305604   2918874    2918874    16386730
     4      7042872    683714     683714     6359158
     5      9887072   1268976    1268976     8618096
     6      6387902    895262     895262     5492640
     7      9989444   2602148    2602148     7387296
     8      9793832   2144596    2144596     7649236
     9      8372776   1012418    1012418     7360358
    10      2858772    499272     499272     2359500
    11      8599090   1403716    1403716     7195374
    12      6457898   2012066    2012066     4445832
    13      5951580    371406     371406     5580174
    14      2955468    651492     651492     2303976
    15      4982618   1014542    1014542     3968076
    16      4273248   1175268    1175268     3097980
    17      4239008    706476     706476     3532532
    18      3588854  

In [17]:
patient_dfs_list[1].head(20)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full
0,2023-09-25 13:00:34.200,flow,52.60,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
1,2023-09-25 13:00:34.200,pressure,10.87,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
2,2023-09-25 13:00:34.400,pressure,11.47,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
3,2023-09-25 13:00:34.400,flow,127.00,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
4,2023-09-25 13:00:34.600,flow,126.70,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
5,2023-09-25 13:00:34.600,pressure,11.39,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
6,2023-09-25 13:00:34.800,flow,105.20,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
7,2023-09-25 13:00:34.800,pressure,12.30,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
8,2023-09-25 13:00:35.000,pressure,5.37,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good
9,2023-09-25 13:00:35.000,flow,54.60,2,0,2023-09-25 13:00:34,2023-10-31 22:17:08,B3A20605023_2023_good


# Remove outliers

In [18]:
import pandas as pd
import numpy as np
from tqdm import tqdm

results = []

for i, df in tqdm(enumerate(patient_dfs_list, start=1), desc="Outlier removal per patient"):
    if not isinstance(df, pd.DataFrame) or df.empty:
        results.append({"index": i, "note": "empty or not DataFrame"})
        continue

    df_clean = df.copy()
    stats = {}

    # Process separately for each test_type (pressure, flow)
    for t in ["pressure", "flow"]:
        sub = df_clean[df_clean["test_type"] == t]
        if sub.empty:
            continue

        mean = sub["test_value"].mean()
        std = sub["test_value"].std()

        # Define cutoff range: [mean - 5*std, mean + 5*std]
        lower = mean - 5 * std
        upper = mean + 5 * std

        mask = (df_clean["test_type"] == t) & ~df_clean["test_value"].between(lower, upper)
        outliers = mask.sum()

        # Drop outliers
        df_clean = df_clean.drop(df_clean[mask].index)

        stats[f"{t}_mean"] = mean
        stats[f"{t}_std"] = std
        stats[f"{t}_lower"] = lower
        stats[f"{t}_upper"] = upper
        stats[f"{t}_removed"] = outliers

    # Update the list with the cleaned frame
    patient_dfs_list[i-1] = df_clean.reset_index(drop=True)

    # Record metadata
    results.append({
        "index": i,
        "patient_id": int(df["patient_id"].iloc[0]) if "patient_id" in df.columns else None,
        "rows_before": len(df),
        "rows_after": len(df_clean),
        **stats
    })

# Build summary DataFrame
summary_outliers = pd.DataFrame(results)

pd.set_option("display.max_rows", None)
print("\nOutlier cleaning summary:")
print(summary_outliers.to_string(index=False))

# Save if needed
OUT_OUTLIER_SUMMARY = Path(r"D:\Medical_AI\Data\processed\outlier_summary.csv")
summary_outliers.to_csv(OUT_OUTLIER_SUMMARY, index=False)
print(f"\nSaved outlier summary to: {OUT_OUTLIER_SUMMARY}")


Outlier removal per patient: 86it [03:23,  2.36s/it]


Outlier cleaning summary:
 index  patient_id  rows_before  rows_after  pressure_mean  pressure_std  pressure_lower  pressure_upper  pressure_removed  flow_mean   flow_std   flow_lower  flow_upper  flow_removed
     1           1      7136450     7135722       6.163113      2.043031       -4.052044       16.378271                 0  37.833218  95.540243  -439.867997  515.534434           728
     2           2      4283030     4274567       6.382108      2.580529       -6.520537       19.284753                 3  62.710057 408.058559 -1977.582736 2103.002851          8460
     3           3     16386730    16215531       8.111420      5.968652      -21.731840       37.954679                 8 163.187794 927.452472 -4474.074564 4800.450151        171191
     4           4      6359158     6358944       6.567872      4.170325      -14.283751       27.419495                 0  28.354085  63.131739  -287.304611  344.012781           214
     5           5      8618096     8617832       8.0

In [21]:
patient_dfs_list[70].shape

(13123389, 8)

In [20]:
for df in patient_dfs_list:
    pid = int(df['patient_id'].iloc[0])
    print(f"Patient {pid}:")
    print(df.iloc[[0]].to_string())
    print()

Patient 1:
                timestamp test_type  test_value  patient_id  label          start_date            end_date      machine_year_full
0 2023-10-02 01:04:20.200      flow       -38.4           1      0 2023-10-02 01:04:20 2023-11-06 08:58:10  B3A20604054_2023_good

Patient 2:
                timestamp test_type  test_value  patient_id  label          start_date            end_date      machine_year_full
0 2023-09-25 13:00:34.200      flow        52.6           2      0 2023-09-25 13:00:34 2023-10-31 22:17:08  B3A20605023_2023_good

Patient 3:
                timestamp test_type  test_value  patient_id  label          start_date            end_date      machine_year_full
0 2023-09-01 15:52:11.200      flow       -38.4           3      0 2023-09-01 15:52:11 2023-10-21 06:45:55  B3A20605070_2023_good

Patient 4:
                timestamp test_type  test_value  patient_id  label          start_date            end_date      machine_year_full
0 2024-12-01 08:11:47.200      flow        

Step 1.4 Add countdown & drop label 0 (optional)

In [43]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import gc

def process_and_cut_7days_inplace(patient_dfs_list):
    """
    For ALL patients (label 0 and label 1):
      1. Coerce & validate timestamps
      2. Cut to last 7*24h window (countdown days 0..6)
      3. Add features relative to first actual recorded timestamp in window
    Mutates patient_dfs_list in place; returns None.
    """
    kept = []

    for i, df in tqdm(enumerate(patient_dfs_list, start=1), desc="Enriching & cutting 7 days"):
        if not isinstance(df, pd.DataFrame) or df.empty or 'label' not in df.columns:
            continue
        if 'timestamp' not in df.columns:
            continue

        # Coerce datetimes
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        if 'start_date' in df.columns:
            df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
        if 'end_date' in df.columns:
            df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

        # Boolean mask to avoid MemoryError from dropna inplace
        need_cols = ['timestamp']
        if 'start_date' in df.columns: need_cols.append('start_date')
        if 'end_date'   in df.columns: need_cols.append('end_date')
        mask = df[need_cols].notna().all(axis=1)
        if not mask.all():
            df = df.loc[mask].reset_index(drop=True)
        if df.empty:
            continue

        # Patient-level end date
        end0 = df['end_date'].dropna().iloc[0] if 'end_date' in df.columns and df['end_date'].notna().any() else pd.NaT
        if pd.isna(end0):
            continue

        # Sort
        df = df.sort_values('timestamp', kind='mergesort').reset_index(drop=True)

        # ✅ +1 second on start_ts so floor((end_ts - start_ts) / 86400) = 6, never 7
        end_ts   = end0
        start_ts = end_ts - pd.Timedelta(days=7) + pd.Timedelta(seconds=1)

        window_mask = (df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)
        df = df.loc[window_mask].reset_index(drop=True)
        if df.empty:
            continue

        ts = df['timestamp']

        # Countdown: floor((end_ts - timestamp) / 86400), guaranteed 0..6
        seconds_left = (end_ts - ts).dt.total_seconds()
        countdown    = np.floor(seconds_left / 86400.0).astype('int32')

        # Sanity check — should never trigger with +1s fix
        cd_min, cd_max = int(countdown.min()), int(countdown.max())
        pid = df['patient_id'].iloc[0] if 'patient_id' in df.columns else i
        if cd_min < 0 or cd_max > 6:
            print(f"⚠️  Patient {pid}: countdown {cd_min}..{cd_max} still out of [0..6] — skipping")
            continue

        # ✅ Use first actual recorded timestamp as reference (patient may not use machine every hour)
        actual_start = ts.min()

        # Features
        df['start_date']                = actual_start
        df['delta_time_seconds']        = ts.diff().dt.total_seconds().fillna(0).clip(lower=0)
        df['time_from_start_seconds']   = (ts - actual_start).dt.total_seconds()  # 0 at first row
        df['patient_span_days']         = (ts.max() - ts.min()).total_seconds() / 86400.0  # actual span
        df['hour_of_day']               = ts.dt.hour
        df['hour_of_day_scaled']        = df['hour_of_day'] / 24.0
        df['countdown_days_target']     = countdown
        df['countdown_days_log_target'] = np.log2(1.0 + countdown.astype('float32'))

        kept.append(df)
        del ts, seconds_left, countdown, actual_start
        gc.collect()

    patient_dfs_list[:] = kept
    del kept
    gc.collect()
    return None

In [44]:
process_and_cut_7days_inplace(patient_dfs_list)
gc.collect()

print(f"Total patients: {len(patient_dfs_list)}")

# Sanity check
label_counts = {0: 0, 1: 0}
for df in patient_dfs_list:
    label_counts[int(df['label'].iloc[0])] += 1
print(f"Label 0: {label_counts[0]} patients, Label 1: {label_counts[1]} patients")

# Peek at one patient
sample = patient_dfs_list[0]
print(f"\n── Patient {sample['patient_id'].iloc[0]} ──")
print(f"  rows          : {len(sample)}")
print(f"  start_date    : {sample['start_date'].iloc[0]}")    # should match first timestamp
print(f"  end_date      : {sample['end_date'].iloc[0]}")
print(f"  span          : {sample['patient_span_days'].iloc[0]:.3f} days")
print(f"  countdown     : {sample['countdown_days_target'].min()}..{sample['countdown_days_target'].max()}")
print(f"  time_from_start first/last: {sample['time_from_start_seconds'].iloc[0]} / {sample['time_from_start_seconds'].iloc[-1]:.1f}")
print(sample[['timestamp', 'start_date', 'countdown_days_target', 'time_from_start_seconds', 'patient_span_days']].head(3))
print("...")
print(sample[['timestamp', 'start_date', 'countdown_days_target', 'time_from_start_seconds', 'patient_span_days']].tail(3))

Enriching & cutting 7 days: 85it [00:53,  1.58it/s]


Total patients: 85
Label 0: 56 patients, Label 1: 29 patients

── Patient 1 ──
  rows          : 1538815
  start_date    : 2023-10-30 22:59:40.200000
  end_date      : 2023-11-06 08:58:10
  span          : 6.416 days
  countdown     : 0..6
  time_from_start first/last: 0.0 / 554309.8
                timestamp              start_date  countdown_days_target  \
0 2023-10-30 22:59:40.200 2023-10-30 22:59:40.200                      6   
1 2023-10-30 22:59:40.200 2023-10-30 22:59:40.200                      6   
2 2023-10-30 22:59:40.400 2023-10-30 22:59:40.200                      6   

   time_from_start_seconds  patient_span_days  
0                      0.0           6.415623  
1                      0.0           6.415623  
2                      0.2           6.415623  
...
                      timestamp              start_date  \
1538812 2023-11-06 08:58:09.800 2023-10-30 22:59:40.200   
1538813 2023-11-06 08:58:10.000 2023-10-30 22:59:40.200   
1538814 2023-11-06 08:58:10.000 2023-

In [45]:
print(len(patient_dfs_list))

85


In [46]:
patient_dfs_list[28].head()

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target
0,2024-12-25 10:03:01.200,pressure,0.0,29,0,2024-12-25 10:03:01.200,2024-12-31 19:12:03,Y2A18105058_2025_good,0.0,0.0,6.381271,10,0.416667,6,2.807355
1,2024-12-25 10:03:01.200,flow,-38.4,29,0,2024-12-25 10:03:01.200,2024-12-31 19:12:03,Y2A18105058_2025_good,0.0,0.0,6.381271,10,0.416667,6,2.807355
2,2024-12-25 10:03:01.400,flow,-38.4,29,0,2024-12-25 10:03:01.200,2024-12-31 19:12:03,Y2A18105058_2025_good,0.2,0.2,6.381271,10,0.416667,6,2.807355
3,2024-12-25 10:03:01.400,pressure,0.0,29,0,2024-12-25 10:03:01.200,2024-12-31 19:12:03,Y2A18105058_2025_good,0.0,0.2,6.381271,10,0.416667,6,2.807355
4,2024-12-25 10:03:01.600,flow,-38.4,29,0,2024-12-25 10:03:01.200,2024-12-31 19:12:03,Y2A18105058_2025_good,0.2,0.4,6.381271,10,0.416667,6,2.807355


# add log for target

In [18]:
patient_dfs_list[1].head(5)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,countdown_days_target,hour_of_day,hour_of_day_scaled,countdown_days_log_target
0,2024-11-01 02:26:18.200,flow,76.60,59,1,2024-11-01 02:26:18,2024-11-30 03:33:36,B3C21201347_2024_bad,0.0,0.2,29.046736,30,2,0.083333,3.433987
1,2024-11-01 02:26:18.200,pressure,10.25,59,1,2024-11-01 02:26:18,2024-11-30 03:33:36,B3C21201347_2024_bad,0.0,0.2,29.046736,30,2,0.083333,3.433987
2,2024-11-01 02:26:18.400,flow,121.60,59,1,2024-11-01 02:26:18,2024-11-30 03:33:36,B3C21201347_2024_bad,0.2,0.4,29.046736,30,2,0.083333,3.433987
3,2024-11-01 02:26:18.400,pressure,16.56,59,1,2024-11-01 02:26:18,2024-11-30 03:33:36,B3C21201347_2024_bad,0.0,0.4,29.046736,30,2,0.083333,3.433987
4,2024-11-01 02:26:18.600,flow,135.50,59,1,2024-11-01 02:26:18,2024-11-30 03:33:36,B3C21201347_2024_bad,0.2,0.6,29.046736,30,2,0.083333,3.433987


Step 1.5 split into train val test based on pre-defined set

In [48]:
import pandas as pd

# Define path
file_path_split = r"D:\Medical_AI\Data\df_split.csv"

# Load the split dataframe
df_split = pd.read_csv(file_path_split)

# Quick check
print(df_split.head(5))


   patient_id  label collection
0           1      0       test
1           2      0      train
2           3      0      train
3           4      0        val
4           5      0        val


In [49]:
import pandas as pd

# 1) Build a mapping: patient_id -> collection
split_map = dict(
    zip(
        df_split['patient_id'].astype(int),
        df_split['collection'].astype(str)
    )
)

# 2) Route each patient's DataFrame into train/val/test lists
train_list, val_list, test_list = [], [], []
missing_in_split = []  # patient_ids not found in df_split

for df in patient_dfs_list:
    if not isinstance(df, pd.DataFrame) or df.empty or 'patient_id' not in df.columns:
        continue

    # patient_id should be constant within each patient's DataFrame; take first row
    pid = int(df['patient_id'].iloc[0])

    coll = split_map.get(pid, None)
    if coll is None:
        missing_in_split.append(pid)
        continue

    if coll == 'train':
        train_list.append(df)
    elif coll == 'val':
        val_list.append(df)
    elif coll == 'test':
        test_list.append(df)
    else:
        # Unknown label in df_split['collection']; skip or handle as needed
        missing_in_split.append(pid)

print(f"Patients routed -> train: {len(train_list)}, val: {len(val_list)}, test: {len(test_list)}")
if missing_in_split:
    print("Patient IDs missing or unknown in df_split:", sorted(set(missing_in_split)))


Patients routed -> train: 48, val: 15, test: 22


In [50]:
df_train = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()
df_val   = pd.concat(val_list,   ignore_index=True) if val_list   else pd.DataFrame()
df_test  = pd.concat(test_list,  ignore_index=True) if test_list  else pd.DataFrame()

print(len(df_train), len(df_val), len(df_test))


70587703 20671364 26861042


In [51]:
df_test.sample(5)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target
20369237,2024-12-28 17:36:59.800,pressure,14.19,56,0,2024-12-24 17:53:44.000,2024-12-31 17:53:43,YSC22408582_2025_good,0.0,344595.8,6.999988,17,0.708333,3,2.000000
14880892,2024-12-26 02:56:48.600,flow,79.00,40,0,2024-12-24 22:29:50.200,2024-12-31 12:55:37,Y2A24306219_2025_good,0.0,102418.4,6.601236,2,0.083333,5,2.584962
9763493,2024-12-31 06:56:53.800,flow,59.70,24,0,2024-12-24 18:54:53.000,2024-12-31 18:54:52,B3CBD801013_2025_good,0.2,561720.8,6.999988,6,0.250000,0,0.000000
14687429,2024-12-25 09:47:42.400,pressure,4.08,40,0,2024-12-24 22:29:50.200,2024-12-31 12:55:37,Y2A24306219_2025_good,0.2,40672.2,6.601236,9,0.375000,6,2.807355
22464327,2021-12-31 16:53:24.400,pressure,5.13,69,1,2021-12-30 07:36:00.200,2022-01-06 06:14:51,Y2A21606379_2023_bad,0.2,119844.2,6.943644,16,0.666667,5,2.584962


In [52]:
df_train.head(5)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target
0,2023-10-25 13:04:44.200,pressure,5.37,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.0,6.383609,13,0.541667,6,2.807355
1,2023-10-25 13:04:44.200,flow,77.70,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.0,6.383609,13,0.541667,6,2.807355
2,2023-10-25 13:04:44.400,flow,-37.80,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.2,0.2,6.383609,13,0.541667,6,2.807355
3,2023-10-25 13:04:44.400,pressure,4.04,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.2,6.383609,13,0.541667,6,2.807355
4,2023-10-25 13:04:44.600,pressure,3.78,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.2,0.4,6.383609,13,0.541667,6,2.807355


Step 1.6 MinMax Scaling

(1) scaling for countdown-days_target(y)

In [53]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Column to scale
TARGET_COL = 'countdown_days_target'
SCALED_COL = f'{TARGET_COL}_scaled'

# 1) Fit on TRAIN only (drop NaNs to avoid fitting issues)
target_scaler = MinMaxScaler()
train_target = df_train[[TARGET_COL]].astype('float64')
target_scaler.fit(train_target.dropna())

# 2) Transform each split
for df in (df_train, df_val, df_test):
    # ensure float dtype; preserve NaNs
    x = df[[TARGET_COL]].astype('float64')
    # transform returns ndarray, align back to index
    df[SCALED_COL] = np.nan
    non_na_mask = x[TARGET_COL].notna()
    if non_na_mask.any():
        df.loc[non_na_mask, SCALED_COL] = target_scaler.transform(x.loc[non_na_mask, [TARGET_COL]])

# Quick check
print(df_train[[TARGET_COL, SCALED_COL]].head())
print(df_val[[TARGET_COL, SCALED_COL]].head())
print(df_test[[TARGET_COL, SCALED_COL]].head())



   countdown_days_target  countdown_days_target_scaled
0                      6                           1.0
1                      6                           1.0
2                      6                           1.0
3                      6                           1.0
4                      6                           1.0
   countdown_days_target  countdown_days_target_scaled
0                      6                           1.0
1                      6                           1.0
2                      6                           1.0
3                      6                           1.0
4                      6                           1.0
   countdown_days_target  countdown_days_target_scaled
0                      6                           1.0
1                      6                           1.0
2                      6                           1.0
3                      6                           1.0
4                      6                           1.0


In [54]:
df_train.head(5)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target,countdown_days_target_scaled
0,2023-10-25 13:04:44.200,pressure,5.37,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.0,6.383609,13,0.541667,6,2.807355,1.0
1,2023-10-25 13:04:44.200,flow,77.70,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.0,6.383609,13,0.541667,6,2.807355,1.0
2,2023-10-25 13:04:44.400,flow,-37.80,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.2,0.2,6.383609,13,0.541667,6,2.807355,1.0
3,2023-10-25 13:04:44.400,pressure,4.04,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.2,6.383609,13,0.541667,6,2.807355,1.0
4,2023-10-25 13:04:44.600,pressure,3.78,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.2,0.4,6.383609,13,0.541667,6,2.807355,1.0


In [55]:
# Print min and max used for scaling
min_val = target_scaler.data_min_[0]
max_val = target_scaler.data_max_[0]

print(f"Min target (days): {min_val}")
print(f"Max target (days): {max_val}")


Min target (days): 0.0
Max target (days): 6.0


In [56]:
df_train.sample(10)


,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target,countdown_days_target_scaled
52351955,2022-12-10 23:36:51.400,pressure,5.86,67,1,2022-12-04 23:25:17.000,2022-12-11 23:25:16,Y2A21104526_2023_bad,0.2,519094.4,6.999988,23,0.958333,0,0.000000,0.000000
23701592,2023-07-10 23:43:05.800,pressure,3.94,41,0,2023-07-04 20:10:57.200,2023-07-11 11:59:59,Y2C16702047_2023_good,0.0,531128.6,6.659049,23,0.958333,0,0.000000,0.000000
6448713,2024-12-24 23:03:39.800,pressure,11.60,8,0,2024-12-24 22:40:42.000,2024-12-31 22:40:41,B3C23C24308_2025_good,0.0,1377.8,6.999988,23,0.958333,6,2.807355,1.000000
27689363,2024-12-30 00:56:04.400,flow,23.00,45,0,2024-12-24 12:53:10.200,2024-12-31 12:51:12,Y2C21A01536_2025_good,0.0,475374.2,6.998632,0,0.000000,1,1.000000,0.166667
40199596,2021-10-11 09:27:30.800,flow,35.50,58,1,2021-10-04 10:37:17.200,2021-10-11 10:04:14,B3320402093_2023_bad,0.2,600613.6,6.977046,9,0.375000,0,0.000000,0.000000
7185160,2024-12-27 11:59:08.800,pressure,6.51,8,0,2024-12-24 22:40:42.000,2024-12-31 22:40:41,B3C23C24308_2025_good,0.2,220706.8,6.999988,11,0.458333,4,2.321928,0.666667
28094553,2023-10-31 00:50:59.600,pressure,5.09,47,0,2023-10-30 22:55:16.200,2023-11-06 06:00:08,Y2D21105246_2023_good,0.2,6943.4,6.295044,0,0.000000,6,2.807355,1.000000
39502472,2021-10-09 09:51:38.000,pressure,10.45,58,1,2021-10-04 10:37:17.200,2021-10-11 10:04:14,B3320402093_2023_bad,0.0,429260.8,6.977046,9,0.375000,2,1.584962,0.333333
69732180,2024-01-26 18:33:18.200,pressure,3.14,84,1,2024-01-22 09:13:50.200,2024-01-28 20:07:57,YSD21503317_2024_bad,0.0,379168.0,6.454245,18,0.750000,2,1.584962,0.333333
61314188,2021-04-09 21:44:03.800,pressure,4.09,79,1,2021-04-07 00:10:40.000,2021-04-14 00:10:39,Y2D20215150_2023_bad,0.2,250403.8,6.999988,21,0.875000,4,2.321928,0.666667


(1.5) scaling for countdown_days_log_target

In [57]:
from sklearn.preprocessing import MinMaxScaler

# Refit the scaler only on valid target range (label ≠ 0 already applied)
log_target_scaler = MinMaxScaler()
log_target_scaler.fit(df_train[['countdown_days_log_target']])  # Automatically uses min/max from real targets

# Apply to all splits
df_train['countdown_days_log_target_scaled'] = log_target_scaler.transform(df_train[['countdown_days_log_target']])
df_val['countdown_days_log_target_scaled'] = log_target_scaler.transform(df_val[['countdown_days_log_target']])
df_test['countdown_days_log_target_scaled'] = log_target_scaler.transform(df_test[['countdown_days_log_target']])

In [58]:
# Print min and max used for scaling
min_val_log = log_target_scaler.data_min_[0]
max_val_log = log_target_scaler.data_max_[0]

print(f"Min target (days): {min_val_log}")
print(f"Max target (days): {max_val_log}")


Min target (days): 0.0
Max target (days): 2.8073549270629883


In [59]:
df_train.sample(10)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target,countdown_days_target_scaled,countdown_days_log_target_scaled
15424233,2023-10-31 09:31:46.800,flow,15.70,27,0,2023-10-30 08:24:36.000,2023-11-06 08:24:35,Y2A17601035_2023_good,0.0,90430.8,6.999988,9,0.375000,5,2.584962,0.833333,0.920782
52175194,2022-12-10 02:21:17.400,flow,3.30,67,1,2022-12-04 23:25:17.000,2022-12-11 23:25:16,Y2A21104526_2023_bad,0.0,442560.4,6.999988,2,0.083333,1,1.000000,0.166667,0.356207
69231042,2024-01-25 13:49:39.400,pressure,4.59,84,1,2024-01-22 09:13:50.200,2024-01-28 20:07:57,YSD21503317_2024_bad,0.0,275749.2,6.454245,13,0.541667,3,2.000000,0.500000,0.712414
20231395,2024-12-24 19:37:44.800,pressure,5.27,35,0,2024-12-24 08:15:17.000,2024-12-31 08:15:16,Y2A21607360_2025_good,0.2,40947.8,6.999988,19,0.791667,6,2.807355,1.000000,1.000000
48943382,2023-05-19 01:08:34.400,flow,12.20,64,1,2023-05-12 18:50:31.200,2023-05-19 08:10:56,Y2A20703184_2023_bad,0.0,541083.2,6.555843,1,0.041667,0,0.000000,0.000000,0.000000
54289125,2023-03-28 01:35:31.200,flow,9.00,70,1,2023-03-26 21:08:30.200,2023-04-02 08:22:29,Y2A21B01199_2023_bad,0.0,102421.0,6.468042,1,0.041667,5,2.584962,0.833333,0.920782
38286783,2021-10-05 10:47:46.800,flow,36.70,58,1,2021-10-04 10:37:17.200,2021-10-11 10:04:14,B3320402093_2023_bad,0.2,87029.6,6.977046,10,0.416667,5,2.584962,0.833333,0.920782
36687627,2023-10-31 21:16:11.000,pressure,5.50,57,0,2023-10-30 20:00:48.200,2023-11-06 03:26:48,YSE23104366_2023_good,0.0,90922.8,6.309720,21,0.875000,5,2.584962,0.833333,0.920782
26919554,2024-12-26 09:56:33.400,pressure,6.00,45,0,2024-12-24 12:53:10.200,2024-12-31 12:51:12,Y2C21A01536_2025_good,0.2,162203.2,6.998632,9,0.375000,5,2.584962,0.833333,0.920782
53500081,2021-10-06 22:54:06.000,pressure,3.87,68,1,2021-10-04 19:42:53.200,2021-10-07 11:12:22,Y2A21106018_2023_bad,0.2,184272.8,2.645472,22,0.916667,0,0.000000,0.000000,0.000000


(2) scaling for  test_value (x)

In [60]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

TYPES = ["pressure", "flow"]
SCALED_COL = "test_value_scaled"

# init output column
for df in (df_train, df_val, df_test):
    df[SCALED_COL] = np.nan

scalers = {}

for t in TYPES:
    # ---- Fit on TRAIN only for this type ----
    m_train = (df_train["test_type"] == t) & df_train["test_value"].notna()
    if not m_train.any():
        # no data of this type in train -> skip this type
        continue

    x_train = df_train.loc[m_train, ["test_value"]].astype("float64")

    # handle zero-variance edge case
    if x_train["test_value"].min() == x_train["test_value"].max():
        scalers[t] = None  # mark as constant; we’ll assign 0.0 later
    else:
        scaler = MinMaxScaler()
        scaler.fit(x_train)
        scalers[t] = scaler

    # ---- Transform each split for this type ----
    for df in (df_train, df_val, df_test):
        m = (df["test_type"] == t) & df["test_value"].notna()
        if not m.any():
            continue

        if scalers[t] is None:
            # constant feature in train -> map to 0.0
            df.loc[m, SCALED_COL] = 0.0
        else:
            vals = df.loc[m, ["test_value"]].astype("float64")
            df.loc[m, SCALED_COL] = scalers[t].transform(vals)
            # optional safety if val/test exceed train min-max
            df.loc[m, SCALED_COL] = df.loc[m, SCALED_COL].clip(0.0, 1.0)

# quick checks
print(df_train[[ "test_type", "test_value", "test_value_scaled"]].head())
print(df_val  [[ "test_type", "test_value", "test_value_scaled"]].head())
print(df_test [[ "test_type", "test_value", "test_value_scaled"]].head())




  test_type  test_value  test_value_scaled
0  pressure        5.37           0.027914
1      flow       77.70           0.486997
2      flow      -37.80           0.002517
3  pressure        4.04           0.021000
4  pressure        3.78           0.019649
  test_type  test_value  test_value_scaled
0  pressure        4.04           0.021000
1      flow       15.70           0.226930
2  pressure        4.09           0.021260
3      flow       16.40           0.229866
4      flow       16.50           0.230285
  test_type  test_value  test_value_scaled
0  pressure        3.91           0.020324
1      flow       67.00           0.442114
2      flow       69.10           0.450923
3  pressure        3.96           0.020584
4  pressure        4.39           0.022819


In [61]:
# Assuming you stored them in a dict called scalers = {"pressure": ..., "flow": ...}

for t, scaler in scalers.items():
    if scaler is None:  # handle constant-value case
        print(f"{t}: constant feature in train (min = max)")
        continue
    min_val = scaler.data_min_[0]
    max_val = scaler.data_max_[0]
    print(f"{t} → min={min_val:.4f}, max={max_val:.4f}")


pressure → min=0.0000, max=192.3800
flow → min=-38.4000, max=200.0000


In [62]:
df_train.sample(10)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target,countdown_days_target_scaled,countdown_days_log_target_scaled,test_value_scaled
15593650,2023-11-01 05:35:51.600,flow,55.40,27,0,2023-10-30 08:24:36.000,2023-11-06 08:24:35,Y2A17601035_2023_good,0.0,162675.6,6.999988,5,0.208333,5,2.584962,0.833333,0.920782,0.393456
12039094,2024-12-27 18:45:52.400,flow,56.90,15,0,2024-12-24 09:50:23.200,2024-12-31 09:50:03,B3C24513203_2025_good,0.0,291329.2,6.999766,18,0.750000,3,2.000000,0.500000,0.712414,0.399748
5668419,2024-12-28 00:38:05.000,flow,8.20,7,0,2024-12-24 13:06:55.200,2024-12-31 13:05:48,B3C23619117_2025_good,0.2,300669.8,6.999222,0,0.000000,3,2.000000,0.500000,0.712414,0.195470
16605881,2023-11-05 08:57:01.800,flow,12.20,27,0,2023-10-30 08:24:36.000,2023-11-06 08:24:35,Y2A17601035_2023_good,0.0,520345.8,6.999988,8,0.333333,0,0.000000,0.000000,0.000000,0.212248
50923413,2022-12-06 01:37:38.400,flow,32.70,67,1,2022-12-04 23:25:17.000,2022-12-11 23:25:16,Y2A21104526_2023_bad,0.2,94341.4,6.999988,1,0.041667,5,2.584962,0.833333,0.920782,0.298238
24530311,2023-06-05 16:24:59.800,flow,23.00,44,0,2023-06-03 12:00:00.000,2023-06-10 11:59:59,Y2C21604310_2023_good,0.0,188699.8,6.999988,16,0.666667,4,2.321928,0.666667,0.827087,0.257550
59968022,2024-04-20 19:31:23.600,pressure,10.20,76,1,2024-04-17 17:42:36.000,2024-04-24 17:42:35,Y2C19408118_2024_bad,0.0,265727.6,6.999988,19,0.791667,3,2.000000,0.500000,0.712414,0.053020
62112645,2021-04-13 00:25:47.000,flow,7.00,79,1,2021-04-07 00:10:40.000,2021-04-14 00:10:39,Y2D20215150_2023_bad,0.2,519307.0,6.999988,0,0.000000,0,0.000000,0.000000,0.000000,0.190436
5385883,2024-12-26 22:10:09.000,pressure,3.78,7,0,2024-12-24 13:06:55.200,2024-12-31 13:05:48,B3C23619117_2025_good,0.2,205393.8,6.999222,22,0.916667,4,2.321928,0.666667,0.827087,0.019649
7502166,2024-12-28 07:54:56.600,flow,26.70,8,0,2024-12-24 22:40:42.000,2024-12-31 22:40:41,B3C23C24308_2025_good,0.2,292454.6,6.999988,7,0.291667,3,2.000000,0.500000,0.712414,0.273070


(3) scaling for  time_from_start_seconds (x)

In [63]:
from sklearn.preprocessing import MinMaxScaler

# Fit on train only
time_from_start_scaler = MinMaxScaler()
time_from_start_scaler.fit(df_train[['time_from_start_seconds']])

# Transform all splits
df_train['time_from_start_scaled'] = time_from_start_scaler.transform(df_train[['time_from_start_seconds']])
df_val['time_from_start_scaled']   = time_from_start_scaler.transform(df_val[['time_from_start_seconds']])
df_test['time_from_start_scaled']  = time_from_start_scaler.transform(df_test[['time_from_start_seconds']])

In [64]:
print("Scaler fitted on time_from_start_seconds from df_train:")
print("Min value:", time_from_start_scaler.data_min_[0])
print("Max value:", time_from_start_scaler.data_max_[0], f"{time_from_start_scaler.data_max_[0]/(3600*24):.2f}","days")


Scaler fitted on time_from_start_seconds from df_train:
Min value: 0.0
Max value: 604799.0 7.00 days


In [65]:
# Add collection tag to each split (fastest)
df_train.loc[:, 'collection'] = 'train'
df_val.loc[:,   'collection'] = 'val'
df_test.loc[:,  'collection'] = 'test'


In [66]:
df_train.head(5)

,timestamp,test_type,test_value,patient_id,label,start_date,end_date,machine_year_full,delta_time_seconds,time_from_start_seconds,patient_span_days,hour_of_day,hour_of_day_scaled,countdown_days_target,countdown_days_log_target,countdown_days_target_scaled,countdown_days_log_target_scaled,test_value_scaled,time_from_start_scaled,collection
0,2023-10-25 13:04:44.200,pressure,5.37,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.0,6.383609,13,0.541667,6,2.807355,1.0,1.0,0.027914,0.000000e+00,train
1,2023-10-25 13:04:44.200,flow,77.70,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.0,6.383609,13,0.541667,6,2.807355,1.0,1.0,0.486997,0.000000e+00,train
2,2023-10-25 13:04:44.400,flow,-37.80,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.2,0.2,6.383609,13,0.541667,6,2.807355,1.0,1.0,0.002517,3.306884e-07,train
3,2023-10-25 13:04:44.400,pressure,4.04,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.0,0.2,6.383609,13,0.541667,6,2.807355,1.0,1.0,0.021000,3.306884e-07,train
4,2023-10-25 13:04:44.600,pressure,3.78,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,B3A20605023_2023_good,0.2,0.4,6.383609,13,0.541667,6,2.807355,1.0,1.0,0.019649,6.613768e-07,train


# save the dfs

In [67]:
import os
import pandas as pd

# Where to save
ROOT = r"D:\Medical_AI\Cleaned\7days_rawdata_parquet"

# Ensure folder
os.makedirs(ROOT, exist_ok=True)

def _prep_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """Optional: light prep for smaller/faster parquet files."""
    d = df.copy(deep=False)
    # Make sure timestamps are datetime
    if 'timestamp' in d.columns:
        d['timestamp'] = pd.to_datetime(d['timestamp'], errors='coerce')
    # Strings with few unique values compress better as category
    for col in ('test_type', 'machine_year_full', 'collection'):
        if col in d.columns and d[col].dtype == object:
            d[col] = d[col].astype('category')
    return d

# ---- Save (single parquet per split) ----
df_train.to_parquet(os.path.join(ROOT, "train.parquet"), engine="pyarrow", index=False)
df_val.to_parquet(  os.path.join(ROOT, "val.parquet"),   engine="pyarrow", index=False)
df_test.to_parquet( os.path.join(ROOT, "test.parquet"),  engine="pyarrow", index=False)


